# Perception dataset readiness

## tl;dr
The available data support the first frozen-package comparison and existing-memory diagnosis. New generalization, causal branches, occlusion and software-use claims need targeted preparation or collection. Large PushT has 18,685 episodes but 185 exact starting-pose groups. No model training or downloads occur here.

## Context & Methods
Read-only metadata and prepared numerical-label checks, collected on 8 September 2026. The exact implementation is [audit_perception_data.py](../scripts/audit_perception_data.py); the report is [Dataset readiness](../docs/perception-data-readiness-2026-09-08.md).

### Key Assumptions
Use the repository and existing dataset locations. Existing held-outs have already been inspected. File presence is not full-media integrity; first-pose grouping is not a complete trajectory-overlap audit. This notebook uses the project Python and its NumPy/h5py dependencies. Cells were executed sequentially with that interpreter; no Jupyter kernel was available in this environment.

## Data
The collector reads current source manifests, prepared episode labels, HDF5 headers/initial poses and video metadata. It leaves source files unchanged. Importing it does not run its command-line entry point.

In [1]:
from pathlib import Path
import json, runpy
ROOT = Path.cwd()
if not (ROOT / 'scripts/audit_perception_data.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts/audit_perception_data.py').is_file()
audit = runpy.run_path(str(ROOT / 'scripts/audit_perception_data.py'), run_name='readiness_notebook')
results = {}
print('Collector:', ROOT / 'scripts/audit_perception_data.py')


Collector: /home/alex/Documents/path-wm/scripts/audit_perception_data.py


## Results
### 1. COCO identities, labels and split groups
All annotation-file ID joins and prepared union-mask identities are checked. Source image decoding and transform alignment are not repeated here.

In [2]:
results['coco'] = audit['coco']()
c = results['coco']
print(json.dumps({k:c[k] for k in ['source_images','split_frames','split_row_overlap','split_duplicate_group_overlap','official_validation_image_directory_present','annotation_files']}, indent=2))
print('Mask identity/value failures:', {s:{k:v for k,v in r.items() if k not in ['images','shape']} for s,r in c['prepared_union_masks'].items()})


{
  "source_images": 82783,
  "split_frames": {
    "test": 4146,
    "train": 74501,
    "validation": 4136
  },
  "split_row_overlap": {
    "test/train": 0,
    "test/validation": 0,
    "train/validation": 0
  },
  "split_duplicate_group_overlap": {
    "test/train": 0,
    "test/validation": 0,
    "train/validation": 0
  },
  "official_validation_image_directory_present": false,
  "annotation_files": {
    "instances_train2014.json": {
      "images": 82783,
      "annotations": 604907,
      "categories": 80,
      "duplicate_annotation_ids": 0,
      "orphan_annotation_image_ids": 0,
      "missing_image_filenames": 0
    },
    "person_keypoints_train2014.json": {
      "images": 82783,
      "annotations": 185316,
      "categories": 1,
      "duplicate_annotation_ids": 0,
      "orphan_annotation_image_ids": 0,
      "missing_image_filenames": 0
    },
    "captions_train2014.json": {
      "images": 82783,
      "annotations": 414113,
      "categories": 0,
      "duplicate

### 2. Prepared trajectories
Check every prepared numerical state/action/time record, without decoding RGB. Paddle groups here are generation seeds; CCHI groups are the recorded configuration groups.

In [3]:
results['paddle'] = audit['prepared_trajectories']('data/paddle/baseline', True)
results['pusht_cchi'] = audit['prepared_trajectories']('data/pusht_world_model/cchi_v1')
for name in ['paddle','pusht_cchi']:
    r=results[name]
    print(name, json.dumps({k:r[k] for k in ['episodes','frames','transitions','group_overlap','failed_episode_checks','event_counts']}, indent=2))


paddle {
  "episodes": {
    "train": 5000,
    "validation": 500,
    "test": 500
  },
  "frames": {
    "train": 184808,
    "validation": 18115,
    "test": 17831
  },
  "transitions": {
    "train": 179808,
    "validation": 17615,
    "test": 17331
  },
  "group_overlap": {
    "train/validation": 0,
    "train/test": 0,
    "validation/test": 0
  },
  "failed_episode_checks": {
    "row_count": 0,
    "transition_count": 0,
    "nonfinite_state_or_action": 0,
    "nonfinite_timestamps": 0,
    "time_order": 0
  },
  "event_counts": {
    "side_wall": 14287,
    "paddle_bound": 5295,
    "paddle_miss": 5979,
    "loss": 5979,
    "paddle_hit": 2039,
    "ceiling": 2018
  }
}
pusht_cchi {
  "episodes": {
    "train": 164,
    "test": 22,
    "validation": 20
  },
  "frames": {
    "train": 20493,
    "test": 2506,
    "validation": 2651
  },
  "transitions": {
    "train": 20329,
    "test": 2484,
    "validation": 2631
  },
  "group_overlap": {
    "train/validation": 0,
    "trai

### 3. Large archives
Read schemas, episode lengths/offsets and exact initial PushT pose groups. These checks establish present availability and repeated starts, not complete decode integrity or simulator replay.

In [4]:
results['large_pusht'] = audit['hdf_inventory']('data/pusht/pusht_expert_train.h5')
results['tworoom'] = audit['hdf_inventory']('data/tworoom/tworoom.h5')
for name in ['large_pusht','tworoom']:
    r=results[name]
    print(name, json.dumps({k:v for k,v in r.items() if k not in ['fields','scope']}, indent=2))


large_pusht {
  "file_bytes": 46300921856,
  "episodes": 18685,
  "frames": 2336736,
  "valid_transitions": 2318051,
  "contiguous_offsets": true,
  "exact_initial_pose_groups": 185,
  "grouping_limit": "Exact first five state values; not a near-configuration or trajectory overlap audit."
}
tworoom {
  "file_bytes": 12775849984,
  "episodes": 10000,
  "frames": 920809,
  "valid_transitions": 910809,
  "contiguous_offsets": true
}


### 4. Passive video/audio metadata
The CSV joins establish referenced-file availability and official subject/location separation. They do not supply executed control commands or verified hidden physical state.

In [5]:
results['natural_videos'] = audit['natural_videos']()
print(json.dumps(results['natural_videos'], indent=2))


{
  "charades": {
    "video_files": 9848,
    "split_rows": {
      "train": 7985,
      "test": 1863
    },
    "missing_by_split": {
      "train": 0,
      "test": 0
    },
    "id_overlap": {
      "train/test": 0
    },
    "subject_overlap": {
      "train/test": 0
    }
  },
  "tau": {
    "metadata_pairs": 12291,
    "audio_files": 12291,
    "video_files": 12291,
    "missing_audio": 0,
    "missing_video": 0,
    "duplicate_pair_rows": 0,
    "split_rows": {
      "train": 8646,
      "test": 3645
    },
    "split_location_counts": {
      "train": 305,
      "test": 126
    },
    "location_overlap": {
      "train/test": 0
    }
  },
  "scope": "CSV identifiers and media-file existence; no full decoding, temporal annotation/synchronization audit or executable-action inference."
}


### 5. Reconcile the report snapshot
This check compares recomputed results with the saved evidence used in the proposal, making stale results visible.

In [6]:
saved=json.loads((ROOT / 'runs/perception_data_cowork_2026-09-08/data_readiness.json').read_text())
for name,value in results.items():
    assert value == saved[name], name + ' differs from the report snapshot'
print('All recomputed sections match the report snapshot.')


All recomputed sections match the report snapshot.


## Takeaways
Use existing prepared data for the narrow first comparison. Prepare crop-aware category labels for a separate semantic-transfer readout. Collect controlled histories/action alternatives for the temporal/causal questions, and a small instrumented application for software use. Keep the current held-outs exploratory and freeze fresh confirmation cases separately. Charades/TAU can support quantitative passive-learning tasks but do not become action-labelled controller data by assigning missing commands to zero.